# Piloto de fine-tuning — español sintético con Chatterbox V2

Genera la parte en español del entrenamiento de cada voz: las **400 frases** de
`piloto/frases_sinteticas.json` (unos 45 min de audio) dichas con la voz del hablante, clonada con
Chatterbox V2 a partir de su referencia `speaker_refs/<voz>/<voz>_ref02.wav`. Chatterbox V2 es el profesor
sin alucinaciones: en la escucha a ciegas sacó 4,25 de acento nativo y 5 de «sin alucinaciones». Parámetros
por defecto (`exaggeration=0.5`, `cfg_weight=0.5`, `temperature=0.8`), los mismos que con Naval.

Salida en Drive: `fine_tunned_naval/piloto/sintetico/<voz>/chatterbox_v2/` (`s001` … `s400`). Naval ya
está hecho (en `piloto/sintetico/chatterbox_v2/`). Después, el cuaderno 03 filtra los clips de cada voz.

**GPU:** T4 (o L4). **Tiempo estimado:** unos 60 min por voz en T4. Se puede hacer una voz por sesión:
pon solo esa en `VOCES`.

Uso: celda 1, *Entorno de ejecución → Reiniciar sesión*, y el resto en orden. Si se corta, vuelve a empezar:
lo que ya esté en Drive no se genera otra vez. La última celda sube todo a Drive y desconecta la GPU.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
# Desde GitHub: la 0.1.7 de PyPI no tiene el parámetro t3_model. torchvision a juego con torch 2.6.0.
# El aviso rojo "pip's dependency resolver..." es normal.
!pip install -q "chatterbox-tts @ git+https://github.com/resemble-ai/chatterbox.git@master" torchvision==0.21.0

In [ ]:
# 2. Parámetros
VOCES = ["nivi", "brett", "deutsch"]   # Naval ya está hecho
PARAMS = dict(exaggeration=0.5, cfg_weight=0.5, temperature=0.8)

In [ ]:
# 3. Preparar: código del repo, Drive, frases y referencias
import os, sys, subprocess, json
from pathlib import Path
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path[:0] = ["/content/repo/criba", "/content/repo/piloto"]
import comun, rutas

comun.montar_drive()
comun.FRASES = Path("/content/repo/piloto/frases_sinteticas.json")
FRASES, SILENCIO_MS = comun.cargar_items()
# Las mismas 400 frases para cada voz; solo cambia la referencia
ITEMS = {voz: [{**it, "voz": voz, "ref": voz} for it in FRASES] for voz in VOCES}
REFS = comun.comprobar([it for items in ITEMS.values() for it in items])
for voz in VOCES:
    hechos = len(list((rutas.sintetico(comun.DRIVE, voz) / "chatterbox_v2").glob("s*.wav")))
    print(f"  {voz}: {hechos} de {len(ITEMS[voz])} clips ya generados")

In [ ]:
# 4. Generar, voz a voz
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
VARIANTE = "chatterbox_v2"
modelo = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v2")

def sintetizar(texto, ref):
    return modelo.generate(texto, language_id="es", **PARAMS), modelo.sr

for voz in VOCES:
    print(f"=== {voz}")
    comun.SALIDA = rutas.sintetico(comun.DRIVE, voz)
    modelo.prepare_conditionals(REFS[voz]["wav"], exaggeration=PARAMS["exaggeration"])
    comun.guardar_entorno(VARIANTE, ["chatterbox-tts", "torch", "transformers", "numpy"])
    comun.generar(VARIANTE, sintetizar, ITEMS[voz], REFS, SILENCIO_MS, params={**PARAMS, "language_id": "es", "voz": voz})

In [ ]:
# 5. Resumen, subir a Drive y desconectar
for voz in VOCES:
    reg = rutas.sintetico(comun.DRIVE, voz) / "chatterbox_v2" / "registro.jsonl"
    filas = [json.loads(l) for l in reg.read_text(encoding="utf-8").splitlines() if l.strip()]
    ok = [f for f in filas if f["ok"]]
    print(f"{voz}: {len(ok)} clips correctos ({sum(f['segundos_audio'] for f in ok) / 60:.0f} min), "
          f"{len(filas) - len(ok)} con error")
comun.desconectar()